---
title: "Exercise bank"
format: html
---

<div id="question-bank" style="display: none !important;">

{{< include ../exercises/exercises.qmd >}}

</div>

<div id="exercise-app-root" class="my-4"></div>

```{=html}
<script src="https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/highlight.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/languages/julia.min.js"></script>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/highlight.js/11.9.0/styles/github.min.css">

<script>
document.addEventListener("DOMContentLoaded", function () {
  const SERVER_SYNC_URL = "https://jack.thomaslabs.co.uk/Math5485-gh/backup.php";
  const SOLUTION_DELAY_MS = 0; // 24 * 60 * 60 * 1000; // 24 hours

  const root = document.getElementById('exercise-app-root');
  const sourcePool = document.getElementById('question-bank');

  if (!root || !sourcePool) return;

  function getFormattedStudentId() {
    let id = localStorage.getItem('studentID');
    if (!id || !/^\d{3}-\d{3}-\d{3}$/.test(id)) {
      const pad = (num) => String(num).padStart(3, '0');
      const p1 = pad(Math.floor(Math.random() * 1000));
      const p2 = pad(Math.floor(Math.random() * 1000));
      const p3 = pad(Math.floor(Math.random() * 1000));
      id = `${p1}-${p2}-${p3}`;
      localStorage.setItem('studentID', id);
    }
    return id;
  }

  let studentID = getFormattedStudentId();
  const allExercises = Array.from(sourcePool.querySelectorAll('.exercise-box'));

  if (allExercises.length === 0) {
    root.innerHTML = '<div class="alert alert-warning">No exercises found.</div>';
    return;
  }

  function cleanTitle(rawText) {
    return rawText.replace(/^(\d+(\.\d+)*\.?\s*|chapter\s+\d+:?\s*|section\s+\d+:?\s*)/i, '').trim();
  }

  let currentLecture = "Exercises";

  allExercises.forEach((el, index) => {
    let prev = el.previousElementSibling;
    while (prev) {
      if (/^H[1-6]$/i.test(prev.tagName) || prev.classList.contains('lecture-title')) {
        currentLecture = cleanTitle(prev.textContent);
        break;
      }
      prev = prev.previousElementSibling;
    }

    if (!el.dataset.lecture) el.dataset.lecture = currentLecture;

    const titleHeader = el.querySelector('h1, h2, h3, h4, .exercise-title, strong');
    const displayLabel = titleHeader ? titleHeader.textContent.trim() : `Ex ${index + 1}`;
    el.dataset.title = displayLabel;

    if (!el.id) {
      const textContent = el.textContent.trim();
      let hash = 0;
      for (let i = 0; i < textContent.length; i++) {
        hash = ((hash << 5) - hash) + textContent.charCodeAt(i);
        hash |= 0;
      }
      el.id = 'exr_hash_' + Math.abs(hash);
    }
  });

  // Inject UI structure
  root.innerHTML = `
    <style>
      .exr-nav-btn {
        display: inline-flex; align-items: center; gap: 6px;
        padding: 7px 16px; border-radius: 999px; font-size: 0.875rem;
        font-weight: 500; border: 1.5px solid; cursor: pointer;
        transition: background 0.15s, box-shadow 0.15s, transform 0.1s;
        box-shadow: 0 1px 3px rgba(0,0,0,0.08);
        background: #fff;
      }
      .exr-nav-btn:hover { box-shadow: 0 3px 8px rgba(0,0,0,0.13); transform: translateY(-1px); }
      .exr-nav-btn:active { transform: translateY(0); box-shadow: 0 1px 2px rgba(0,0,0,0.1); }
      .exr-nav-btn.prev  { border-color: #6c757d; color: #6c757d; }
      .exr-nav-btn.prev:hover  { background: #6c757d; color: #fff; }
      .exr-nav-btn.next  { border-color: #0d6efd; color: #0d6efd; }
      .exr-nav-btn.next:hover  { background: #0d6efd; color: #fff; }
      .exr-nav-btn.rand  { border-color: #0d6efd; background: #0d6efd; color: #fff; }
      .exr-nav-btn.rand:hover  { background: #0b5ed7; border-color: #0b5ed7; }
      .exr-nav-btn svg { flex-shrink: 0; }
      .exr-nav-btn.hint { border-color: #f0ad4e; color: #b07800; }
      .exr-nav-btn.hint:hover { background: #f0ad4e; color: #fff; }
      .exr-nav-btn.hint.hint-on { background: #f0ad4e; color: #fff; border-color: #f0ad4e; }
      #hint-container { border: none; background: transparent; border-radius: 0; padding: 0; margin-top: 0.4rem; }
      #solution-container { border: none; background: #f0faf4; border-radius: 8px; padding: 1rem 1.25rem; margin-top: 0.75rem; }
    </style>
    <div class="card p-3 my-3">
      <div class="d-flex align-items-center justify-content-between flex-wrap gap-3">
        <div class="d-flex gap-2" role="group">
          <button id="prev-exr-btn" class="exr-nav-btn prev">
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M11.354 1.646a.5.5 0 0 1 0 .708L5.707 8l5.647 5.646a.5.5 0 0 1-.708.708l-6-6a.5.5 0 0 1 0-.708l6-6a.5.5 0 0 1 .708 0z"/></svg>
            Previous
          </button>
          <button id="next-exr-btn" class="exr-nav-btn next">
            Next
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M4.646 1.646a.5.5 0 0 1 .708 0l6 6a.5.5 0 0 1 0 .708l-6 6a.5.5 0 0 1-.708-.708L10.293 8 4.646 2.354a.5.5 0 0 1 0-.708z"/></svg>
          </button>
          <button id="random-exr-btn" class="exr-nav-btn rand">
            <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path fill-rule="evenodd" d="M0 3.5A.5.5 0 0 1 .5 3H1c2.202 0 3.827 1.24 4.874 2.418.49.552.865 1.102 1.126 1.532.26-.43.636-.98 1.126-1.532C9.173 4.24 10.798 3 13 3v1c-1.798 0-3.173 1.01-4.126 2.082A9.6 9.6 0 0 0 7.556 8a9.6 9.6 0 0 0 1.317 1.918C9.828 10.99 11.204 12 13 12v1c-2.202 0-3.827-1.24-4.874-2.418A10.6 10.6 0 0 1 7 9.05c-.26.43-.636.98-1.126 1.532C4.827 11.76 3.202 13 1 13H.5a.5.5 0 0 1 0-1H1c1.798 0 3.173-1.01 4.126-2.082A9.6 9.6 0 0 0 6.443 8a9.6 9.6 0 0 0-1.317-1.918C4.172 5.01 2.796 4 1 4H.5a.5.5 0 0 1-.5-.5z"/><path d="M13 5.466V1.534a.25.25 0 0 1 .41-.192l2.36 1.966c.12.1.12.284 0 .384l-2.36 1.966a.25.25 0 0 1-.41-.192zm0 9v-3.932a.25.25 0 0 1 .41-.192l2.36 1.966c.12.1.12.284 0 .384l-2.36 1.966a.25.25 0 0 1-.41-.192z"/></svg>
            Shuffle
          </button>
        </div>

        <div class="d-flex align-items-center gap-2">
          <select id="filter-select" class="form-select form-select-sm" style="width: auto;">
            <option value="all">All questions</option>
            <option value="unrated">❓ Unrated only</option>
            <option value="down">⚠️ Needs review only</option>
            <option value="up">✓ Completed only</option>
          </select>
        </div>
      </div>

      <div class="d-flex justify-content-between align-items-center gap-2 mt-3 pt-2 border-top small text-muted flex-wrap">
        <div>
          <span>Total: <strong id="cnt-all">0</strong></span> |
          <span>Unrated: <strong id="cnt-unrated" class="text-secondary">0</strong></span> |
          <span>Completed: <strong id="cnt-up" class="text-success">0</strong></span> |
          <span>Needs review: <strong id="cnt-down" class="text-warning">0</strong></span>
        </div>
        <div class="d-flex align-items-center gap-2">
          <span id="sync-status" class="badge bg-light text-muted border">Synced</span>
          <span id="streak-display" title="Day streak" style="font-size:.82rem;color:#6c757d;"></span>
          <button id="manage-id-btn" class="btn btn-sm btn-outline-secondary py-0" title="View or change Student ID">🔑 ID</button>
        </div>
      </div>
    </div>

    <!-- Container for active exercise -->
    <div id="random-exercise-display"></div>

    <!-- Hints & Solutions Control Bar -->
    <div id="solution-controls" class="my-3" style="display:none;">
      <div class="d-flex gap-2 flex-wrap align-items-center">
        <button id="btn-show-hint" class="exr-nav-btn hint" title="Toggle hint" style="display:none;">
          <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M2 6a6 6 0 1 1 10.174 4.31c-.203.196-.359.4-.453.619l-.762 1.769A.5.5 0 0 1 10.5 13h-5a.5.5 0 0 1-.46-.302l-.761-1.77a2 2 0 0 0-.453-.618A5.98 5.98 0 0 1 2 6zm3 8.5a.5.5 0 0 1 .5-.5h5a.5.5 0 0 1 0 1l-.224.447a1 1 0 0 1-.894.553H6.618a1 1 0 0 1-.894-.553L5.5 15a.5.5 0 0 1-.5-.5z"/></svg>
          Hint
        </button>
        <button id="btn-show-solution" class="exr-nav-btn" disabled style="display:none; border-color:#6c757d; color:#6c757d;">
          <svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M8 1a2 2 0 0 1 2 2v4H6V3a2 2 0 0 1 2-2zm3 6V3a3 3 0 0 0-6 0v4a2 2 0 0 0-2 2v5a2 2 0 0 0 2 2h6a2 2 0 0 0 2-2V9a2 2 0 0 0-2-2z"/></svg>
          Solution (<span id="solution-timer">30</span>s)
        </button>
      </div>

      <div id="hint-container" style="display:none;"></div>
      <div id="solution-container" style="display:none;"></div>
    </div>

    <!-- Feedback Bar -->
    <div id="feedback-bar" class="mt-3 p-3 bg-light border rounded d-flex flex-wrap align-items-center justify-content-between gap-3" style="display:none;">
      <div class="d-flex align-items-center gap-2">
        <button id="btn-thumbs-up" class="btn btn-outline-success btn-sm">✓ Completed</button>
        <button id="btn-thumbs-down" class="btn btn-outline-warning btn-sm">⚠️ Needs review</button>
      </div>

      <div class="d-flex align-items-center gap-2">
        <span class="small fw-bold text-muted">Difficulty:</span>
        <div id="star-rating" class="d-flex gap-1 fs-5 text-warning" style="cursor: pointer;">
          <span data-star="1">☆</span>
          <span data-star="2">☆</span>
          <span data-star="3">☆</span>
          <span data-star="4">☆</span>
          <span data-star="5">☆</span>
        </div>
      </div>
    </div>
  `;

  // DOM Elements
  const prevBtn = document.getElementById('prev-exr-btn');
  const nextBtn = document.getElementById('next-exr-btn');
  const randomBtn = document.getElementById('random-exr-btn');
  const filterSelect = document.getElementById('filter-select');
  const displayContainer = document.getElementById('random-exercise-display');
  const feedbackBar = document.getElementById('feedback-bar');
  const btnUp = document.getElementById('btn-thumbs-up');
  const btnDown = document.getElementById('btn-thumbs-down');
  const starContainer = document.getElementById('star-rating');
  const manageIdBtn = document.getElementById('manage-id-btn');
  const syncStatus = document.getElementById('sync-status');

  const solutionControls = document.getElementById('solution-controls');
  const btnShowHint = document.getElementById('btn-show-hint');
  const btnShowSolution = document.getElementById('btn-show-solution');
  const solutionTimerSpan = document.getElementById('solution-timer');
  const hintContainer = document.getElementById('hint-container');
  const solutionContainer = document.getElementById('solution-container');

  const cntAll = document.getElementById('cnt-all');
  const cntUnrated = document.getElementById('cnt-unrated');
  const cntUp = document.getElementById('cnt-up');
  const cntDown = document.getElementById('cnt-down');

  let activeExerciseId = null;
  let activeLectureTitle = null;
  let showingTitleCard = false;
  let questionStartTime = null;
  let solutionTimerInterval = null;

  function getVote(id) { return localStorage.getItem('vote_' + id); }
  function getDifficulty(id) { return localStorage.getItem('diff_' + id); }
  function getTimeSpent(id) { return localStorage.getItem('timeSpent_' + id); }
  function getFirstSeen(id) { return localStorage.getItem('firstSeen_' + id); }
  function recordFirstSeen(id) {
    if (!getFirstSeen(id)) {
      localStorage.setItem('firstSeen_' + id, Date.now());
      updateStreakDisplay();
    }
  }
  function formatTimeRemaining(ms) {
    const h = Math.floor(ms / 3600000);
    const m = Math.floor((ms % 3600000) / 60000);
    if (h > 0) return `${h}h ${m}m`;
    return `${m}m`;
  }

  function computeStreak() {
    // Collect all unique days on which any question was first seen
    const daySet = new Set();
    allExercises.forEach(el => {
      const ts = getFirstSeen(el.id);
      if (ts) {
        const d = new Date(parseInt(ts));
        daySet.add(`${d.getFullYear()}-${d.getMonth()}-${d.getDate()}`);
      }
    });
    if (daySet.size === 0) return 0;

    // Sort days and count consecutive streak ending today (or yesterday)
    const today = new Date();
    const toKey = d => `${d.getFullYear()}-${d.getMonth()}-${d.getDate()}`;
    let streak = 0;
    let check = new Date(today);
    // Allow streak to still show if the student hasn't opened anything today yet
    if (!daySet.has(toKey(check))) check.setDate(check.getDate() - 1);
    while (daySet.has(toKey(check))) {
      streak++;
      check.setDate(check.getDate() - 1);
    }
    return streak;
  }

  function updateStreakDisplay() {
    const streak = computeStreak();
    const el = document.getElementById('streak-display');
    if (!el) return;
    if (streak === 0) { el.textContent = ''; return; }
    const flame = streak >= 3 ? '🔥' : '📅';
    el.textContent = `${flame} ${streak}d`;
    el.title = streak === 1 ? '1 day streak' : `${streak} day streak`;
  }

  async function syncProgressToServer(actionMetaData = null) {
    syncStatus.textContent = 'Syncing...';
    syncStatus.className = 'badge bg-warning text-dark border';

    const votes = {};
    const difficulties = {};
    const titles = {};
    const timeSpent = {};
    const firstSeen = {};

    allExercises.forEach(el => {
      const v = getVote(el.id);
      const d = getDifficulty(el.id);
      const t = getTimeSpent(el.id);
      const f = getFirstSeen(el.id);
      if (v) votes[el.id] = v;
      if (d) difficulties[el.id] = d;
      if (t) timeSpent[el.id] = parseInt(t, 10);
      if (f) firstSeen[el.id] = parseInt(f, 10);
      titles[el.id] = el.dataset.title;
    });

    const payload = {
      studentID: localStorage.getItem('studentID'),
      timestamp: new Date().toISOString(),
      votes: votes,
      difficulties: difficulties,
      timeSpentSeconds: timeSpent,
      firstSeenTimestamps: firstSeen,
      exerciseTitles: titles
    };

    if (actionMetaData) payload.lastAction = actionMetaData;

    try {
      const response = await fetch(SERVER_SYNC_URL, {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify(payload)
      });

      if (response.ok) {
        syncStatus.textContent = 'Synced';
        syncStatus.className = 'badge bg-light text-success border';
      }
    } catch (err) {
      syncStatus.textContent = 'Offline';
      syncStatus.className = 'badge bg-light text-secondary border';
    }
  }

  function updateCounters() {
    const total = allExercises.length;
    const upCount = allExercises.filter(el => getVote(el.id) === 'up').length;
    const downCount = allExercises.filter(el => getVote(el.id) === 'down').length;

    cntAll.textContent = total;
    cntUnrated.textContent = total - (upCount + downCount);
    cntUp.textContent = upCount;
    cntDown.textContent = downCount;
  }

  // Track current position as an index into allExercises (avoids findIndex bugs)
  let currentExerciseIndex = -1;

  function getActivePool() {
    const filter = filterSelect.value;
    if (filter === 'unrated') return allExercises.filter(el => !getVote(el.id));
    if (filter === 'down') return allExercises.filter(el => getVote(el.id) === 'down');
    if (filter === 'up') return allExercises.filter(el => getVote(el.id) === 'up');
    return allExercises;
  }

  function updateFeedbackUI() {
    if (!activeExerciseId || showingTitleCard) return;

    const vote = getVote(activeExerciseId);
    btnUp.className = vote === 'up' ? 'btn btn-success btn-sm' : 'btn btn-outline-success btn-sm';
    btnDown.className = vote === 'down' ? 'btn btn-warning btn-sm' : 'btn btn-outline-warning btn-sm';

    const currentDiff = parseInt(getDifficulty(activeExerciseId)) || 0;
    Array.from(starContainer.children).forEach(star => {
      const starVal = parseInt(star.dataset.star);
      star.textContent = starVal <= currentDiff ? '★' : '☆';
    });
  }

  function setupSolutionControls(exerciseElement) {
    clearInterval(solutionTimerInterval);
    
    hintContainer.style.display = 'block';
    solutionContainer.style.display = 'none';
    hintContainer.innerHTML = '';
    solutionContainer.innerHTML = '';
    btnShowHint.classList.remove('hint-on');
    btnShowSolution.classList.remove('next');

    const hintEl = exerciseElement.querySelector('.exercise-hint');
    const solutionEl = exerciseElement.querySelector('.exercise-solution');

    if (!hintEl && !solutionEl) {
      solutionControls.style.display = 'none';
      return;
    }

    solutionControls.style.display = 'block';

    // Configure Hint Button — content rendered into hintContainer below the buttons
    if (hintEl) {
      btnShowHint.style.display = 'inline-flex';
      hintContainer.innerHTML = hintEl.outerHTML;
      hintContainer.querySelector('.exercise-hint').style.display = 'none';
    } else {
      btnShowHint.style.display = 'none';
    }

    // Configure Solution Button (unlocks 24h after first view)
    if (solutionEl) {
      btnShowSolution.style.display = 'inline-flex';
      solutionContainer.innerHTML = solutionEl.innerHTML;

      const firstSeen = parseInt(getFirstSeen(exerciseElement.id) || '0', 10);
      const elapsed = firstSeen ? Date.now() - firstSeen : 0;
      const remaining = SOLUTION_DELAY_MS - elapsed;

      const lockSVG = '<svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M8 1a2 2 0 0 1 2 2v4H6V3a2 2 0 0 1 2-2zm3 6V3a3 3 0 0 0-6 0v4a2 2 0 0 0-2 2v5a2 2 0 0 0 2 2h6a2 2 0 0 0 2-2V9a2 2 0 0 0-2-2z"/></svg>';
      const unlockSVG = '<svg width="14" height="14" viewBox="0 0 16 16" fill="currentColor"><path d="M11 1a2 2 0 0 0-2 2v4a2 2 0 0 1 2 2v5a2 2 0 0 1-2 2H3a2 2 0 0 1-2-2V9a2 2 0 0 1 2-2h5V3a3 3 0 0 1 6 0v4a.5.5 0 0 1-1 0V3a2 2 0 0 0-2-2z"/></svg>';

      if (remaining <= 0) {
        // Already unlocked
        btnShowSolution.disabled = false;
        btnShowSolution.className = 'exr-nav-btn next';
        btnShowSolution.style.cssText = '';
        btnShowSolution.innerHTML = unlockSVG + ' Show Solution';
      } else {
        // Locked — show time remaining, no live countdown needed
        btnShowSolution.disabled = true;
        btnShowSolution.className = 'exr-nav-btn';
        btnShowSolution.style.cssText = 'border-color:#6c757d; color:#6c757d; opacity:0.6; cursor:not-allowed;';
        btnShowSolution.innerHTML = lockSVG + ' Solution in ' + formatTimeRemaining(remaining);
      }
    } else {
      btnShowSolution.style.display = 'none';
    }
  }

  function renderExercise(chosenElement) {
    if (!chosenElement) {
      displayContainer.innerHTML = `<div class="alert alert-info m-0">No exercises found for this filter.</div>`;
      feedbackBar.style.display = 'none';
      solutionControls.style.display = 'none';
      activeExerciseId = null;
      showingTitleCard = false;
      questionStartTime = null;
      return;
    }

    showingTitleCard = false;
    feedbackBar.style.display = 'flex';
    activeExerciseId = chosenElement.id;
    activeLectureTitle = chosenElement.dataset.lecture;
    currentExerciseIndex = allExercises.indexOf(chosenElement);
    questionStartTime = Date.now();
    recordFirstSeen(chosenElement.id);

    const clone = chosenElement.cloneNode(true);
    clone.style.display = 'block';

    // Keep only .exercise-solution stripped from clone; hint is shown below via button
    clone.querySelectorAll('.exercise-hint, .exercise-solution').forEach(e => e.remove());

    displayContainer.innerHTML = '';
    displayContainer.appendChild(clone);

    // Setup hint and solution triggers (must be after clone is in DOM)
    setupSolutionControls(chosenElement);

    if (window.MathJax && window.MathJax.typesetPromise) {
      window.MathJax.typesetPromise([displayContainer]);
    }

    updateFeedbackUI();
  }

  // Hint Click Handler — toggles .exercise-hint rendered below the buttons
  btnShowHint.addEventListener('click', () => {
    const inlineHint = hintContainer.querySelector('.exercise-hint');
    if (!inlineHint) return;
    const isVisible = inlineHint.style.display !== 'none';
    if (isVisible) {
      inlineHint.style.display = 'none';
      btnShowHint.classList.remove('hint-on');
    } else {
      inlineHint.style.display = '';
      btnShowHint.classList.add('hint-on');
      if (window.MathJax && window.MathJax.typesetPromise) {
        window.MathJax.typesetPromise([inlineHint]);
      }
      syncProgressToServer({ exerciseID: activeExerciseId, requestedHint: true });
    }
  });

  // Solution Click Handler — toggles solution visibility and active pill state
  btnShowSolution.addEventListener('click', () => {
    const isVisible = solutionContainer.style.display === 'block';
    if (isVisible) {
      solutionContainer.style.display = 'none';
      btnShowSolution.classList.remove('next');
      btnShowSolution.style.cssText = 'border-color:#0d6efd; color:#0d6efd;';
    } else {
      solutionContainer.style.display = 'block';
      btnShowSolution.classList.add('next');
      btnShowSolution.style.cssText = '';
      if (window.hljs) hljs.highlightAll();
      if (window.MathJax && window.MathJax.typesetPromise) {
        window.MathJax.typesetPromise([solutionContainer]);
      }
      syncProgressToServer({ exerciseID: activeExerciseId, requestedSolution: true });
    }
  });

  function showNextExercise() {
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    // Find where current exercise sits in the pool by index in allExercises
    const posInPool = pool.findIndex(el => allExercises.indexOf(el) === currentExerciseIndex);
    const nextPos = posInPool === -1 ? 0 : (posInPool + 1) % pool.length;
    renderExercise(pool[nextPos]);
  }

  function showPrevExercise() {
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    const posInPool = pool.findIndex(el => allExercises.indexOf(el) === currentExerciseIndex);
    const prevPos = posInPool === -1 ? pool.length - 1 : (posInPool - 1 + pool.length) % pool.length;
    renderExercise(pool[prevPos]);
  }

  function showRandomExercise() {
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    const chosen = pool.length > 1 ? pool.filter(el => el.id !== activeExerciseId)[Math.floor(Math.random() * (pool.length - 1))] : pool[0];
    renderExercise(chosen);
  }

  function handleStatusClick(newVoteState) {
    if (!activeExerciseId || showingTitleCard) return;

    let timeSpentSecs = null;
    if (questionStartTime) {
      const elapsedSeconds = Math.round((Date.now() - questionStartTime) / 1000);
      if (elapsedSeconds > 10) {
        timeSpentSecs = elapsedSeconds;
        const existingTime = parseInt(getTimeSpent(activeExerciseId) || '0', 10);
        localStorage.setItem('timeSpent_' + activeExerciseId, existingTime + timeSpentSecs);
      }
    }

    const currentVote = getVote(activeExerciseId);
    currentVote === newVoteState 
      ? localStorage.removeItem('vote_' + activeExerciseId) 
      : localStorage.setItem('vote_' + activeExerciseId, newVoteState);

    updateFeedbackUI();
    updateCounters();
    syncProgressToServer({ exerciseID: activeExerciseId, statusMarked: newVoteState, timeSpentOnThisSession: timeSpentSecs });
  }

  btnUp.addEventListener('click', () => handleStatusClick('up'));
  btnDown.addEventListener('click', () => handleStatusClick('down'));

  starContainer.addEventListener('click', (e) => {
    if (!activeExerciseId || showingTitleCard) return;
    const starVal = e.target.dataset.star;
    if (!starVal) return;
    getDifficulty(activeExerciseId) === starVal ? localStorage.removeItem('diff_' + activeExerciseId) : localStorage.setItem('diff_' + activeExerciseId, starVal);
    updateFeedbackUI();
    syncProgressToServer();
  });

  filterSelect.addEventListener('change', () => {
    // When filter changes, reset to first exercise in the new pool
    // (not 'next from current' which breaks when current isn't in new pool)
    const pool = getActivePool();
    if (pool.length === 0) return renderExercise(null);
    // If the current exercise is in the new pool, keep it; otherwise go to first
    const stillInPool = pool.find(el => el.id === activeExerciseId);
    renderExercise(stillInPool || pool[0]);
  });
  prevBtn.addEventListener('click', showPrevExercise);
  nextBtn.addEventListener('click', showNextExercise);
  randomBtn.addEventListener('click', showRandomExercise);

  async function fetchAndRestoreProgress(id) {
    syncStatus.textContent = 'Syncing...';
    syncStatus.className = 'badge bg-warning text-dark border';
    try {
      const response = await fetch(`${SERVER_SYNC_URL}?studentID=${encodeURIComponent(id)}`);
      if (!response.ok) throw new Error('Not found');
      const data = await response.json();
      if (data.status === 'error') throw new Error(data.message);

      // Merge server data into localStorage — server wins on conflicts
      let restored = 0;
      Object.entries(data.votes || {}).forEach(([k, v]) => { localStorage.setItem('vote_' + k, v); restored++; });
      Object.entries(data.difficulties || {}).forEach(([k, v]) => { localStorage.setItem('diff_' + k, v); });
      Object.entries(data.timeSpentSeconds || {}).forEach(([k, v]) => { localStorage.setItem('timeSpent_' + k, v); });
      Object.entries(data.firstSeenTimestamps || {}).forEach(([k, v]) => { localStorage.setItem('firstSeen_' + k, v); });

      // Update all UI that reads from localStorage
      updateCounters();
      updateStreakDisplay();
      updateFeedbackUI();
      // Re-render current exercise so solution lock reflects restored firstSeen
      if (currentExerciseIndex >= 0) renderExercise(allExercises[currentExerciseIndex]);

      syncStatus.textContent = 'Synced';
      syncStatus.className = 'badge bg-success text-white border';
      alert(`Progress restored from server (${restored} exercise(s) synced).`);
    } catch (e) {
      syncStatus.textContent = 'Offline';
      syncStatus.className = 'badge bg-danger text-white border';
      alert('Could not fetch progress from server: ' + e.message);
    }
  }

  manageIdBtn.addEventListener('click', () => {
    const currentID = localStorage.getItem('studentID') || '(none)';
    const newID = window.prompt(
      `Your current Student ID is:\n\n  ${currentID}\n\nTo sync progress from another device, paste that device's ID below and press OK.\nLeave blank to keep your current ID.`
    );
    if (newID === null) return;
    const trimmed = newID.trim();
    if (trimmed === '') return;
    if (!/^\d{3}-\d{3}-\d{3}$/.test(trimmed)) {
      alert('Invalid ID format. Expected xxx-xxx-xxx (e.g. 042-317-891).');
      return;
    }
    localStorage.setItem('studentID', trimmed);
    studentID = trimmed;
    // Fetch existing progress for this ID from server, then push local data up
    fetchAndRestoreProgress(trimmed).then(() => syncProgressToServer({ changedID: true }));
  });

  updateCounters();
  updateStreakDisplay();

  // If the URL contains a hash matching a bank exercise id, show that exercise;
  // otherwise start with a random one as usual.
  // Links from lectures use: random-2.html#bank-exr-pi-artan
  const hashId = window.location.hash.slice(1); // e.g. 'bank-exr-pi-artan'
  const hashMatch = hashId ? allExercises.find(el => el.id === hashId) : null;
  if (hashMatch) {
    renderExercise(hashMatch);
  } else {
    showRandomExercise();
  }
});
</script>